Ishan Bashyal 240530/15939241

Importing All Necessary Imports

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.sql.functions import col, sum as spark_sum, when, mean

In [2]:
spark = SparkSession.builder.appName("Student Data Analysis").master("local[*]").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/25 22:22:07 WARN Utils: Your hostname, Inishs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.254.11 instead (on interface en0)
26/05/25 22:22:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 22:22:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/25 22:22:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
# Q1. Load students_data.csv into a DataFrame using inferSchema=True. Print the schema and display the first 5 rows.

df = spark.read.csv("students_data.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- student_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- gpa: double (nullable = true)
 |-- scholarship: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)

+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|student_id|         name|gender|age|      department|year| gpa|scholarship|     city|               email|
+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|       101| Aarav Sharma|  Male| 20|Computer Science|   2| 3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       102|  Priya Thapa|Female| 22|        Business|   4| 3.2|         No|  Pokhara| priya.thapa@uni.edu|
|       103|  Rohan Karki|  Male| 21|     Engineering|   3|NULL|       

In [4]:
# Q2. How many rows and columns does the dataset have? Print both counts.

print("Row count   :", df.count())
print("Column count:", len(df.columns))

Row count   : 25
Column count: 10


In [5]:
''' Q3. Redefine the schema manually using StructType. 
Set gpa as FloatType and student_id as IntegerType,then reload the CSV with this schema.'''

schema = StructType([
    StructField("student_id",  IntegerType(), nullable=True),
    StructField("name",        StringType(),  nullable=True),
    StructField("gender",      StringType(),  nullable=True),
    StructField("age",         IntegerType(), nullable=True),
    StructField("department",  StringType(),  nullable=True),
    StructField("year",        IntegerType(), nullable=True),
    StructField("gpa",         FloatType(),   nullable=True),
    StructField("scholarship", StringType(),  nullable=True),
    StructField("city",        StringType(),  nullable=True),
    StructField("email",       StringType(),  nullable=True)
])

df = spark.read.csv("students_data.csv", header=True, schema=schema)
df.printSchema()
df.show(5)

root
 |-- student_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- gpa: float (nullable = true)
 |-- scholarship: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)

+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|student_id|         name|gender|age|      department|year| gpa|scholarship|     city|               email|
+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|       101| Aarav Sharma|  Male| 20|Computer Science|   2| 3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       102|  Priya Thapa|Female| 22|        Business|   4| 3.2|         No|  Pokhara| priya.thapa@uni.edu|
|       103|  Rohan Karki|  Male| 21|     Engineering|   3|NULL|        

In [6]:
# Q4. Count the number of null values in each column. Which columns have missing data?

null_count = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

null_count.show()

+----------+----+------+---+----------+----+---+-----------+----+-----+
|student_id|name|gender|age|department|year|gpa|scholarship|city|email|
+----------+----+------+---+----------+----+---+-----------+----+-----+
|         0|   0|     0|  2|         1|   0|  3|          0|   0|    0|
+----------+----+------+---+----------+----+---+-----------+----+-----+



In [7]:
# Q5. Filter and display only the rows that have a null value in the gpa column.

df.filter(col("gpa").isNull()).show()

+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+
|student_id|          name|gender|age| department|year| gpa|scholarship|    city|               email|
+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+
|       103|   Rohan Karki|  Male| 21|Engineering|   3|NULL|         No|Lalitpur| rohan.karki@uni.edu|
|       110|  Nisha Tamang|Female| 20|       Arts|   2|NULL|         No|Lalitpur|nisha.tamang@uni.edu|
|       122|Dipika Niroula|Female| 20|   Business|   2|NULL|         No|  Dharan|dipika.niroula@un...|
+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+



In [8]:
# Q6. Fill null values in gpa with the mean GPA. Fill null department with Undeclared and null age with 20.

mean_gpa = df.select(mean(col("gpa"))).collect()[0][0]
print("Mean GPA:", mean_gpa)

df_filled = df.fillna({
    "gpa":        mean_gpa,
    "department": "Undeclared",
    "age":        20
})

df_filled.show()

Mean GPA: 3.304545456712896
+----------+----------------+------+---+----------------+----+---------+-----------+---------+--------------------+
|student_id|            name|gender|age|      department|year|      gpa|scholarship|     city|               email|
+----------+----------------+------+---+----------------+----+---------+-----------+---------+--------------------+
|       101|    Aarav Sharma|  Male| 20|Computer Science|   2|      3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       102|     Priya Thapa|Female| 22|        Business|   4|      3.2|         No|  Pokhara| priya.thapa@uni.edu|
|       103|     Rohan Karki|  Male| 21|     Engineering|   3|3.3045454|         No| Lalitpur| rohan.karki@uni.edu|
|       104|        Sita Rai|Female| 19|Computer Science|   1|      3.9|        Yes|Kathmandu|    sita.rai@uni.edu|
|       105|   Bikash Gurung|  Male| 23|        Business|   4|      2.7|         No|   Butwal|bikash.gurung@uni...|
|       106|  Anita Shrestha|Female| 20|    

In [9]:
# Q7. Select only name, department, and gpa. Display the result.

df_filled.select(
    col("name"),
    col("department"),
    col("gpa")
).show()

+----------------+----------------+---------+
|            name|      department|      gpa|
+----------------+----------------+---------+
|    Aarav Sharma|Computer Science|      3.8|
|     Priya Thapa|        Business|      3.2|
|     Rohan Karki|     Engineering|3.3045454|
|        Sita Rai|Computer Science|      3.9|
|   Bikash Gurung|        Business|      2.7|
|  Anita Shrestha|     Engineering|      3.5|
|   Deepak Pandey|            Arts|      3.1|
|     Kavya Joshi|Computer Science|      3.7|
|    Suresh Limbu|     Engineering|      2.9|
|    Nisha Tamang|            Arts|3.3045454|
|   Prakash Bista|        Business|      3.4|
|     Manisha Oli|Computer Science|      3.6|
| Rajesh Adhikari|     Engineering|      3.0|
|    Sunita Magar|            Arts|      2.8|
|    Kiran Poudel|Computer Science|      3.3|
|     Rupa Basnet|      Undeclared|      3.7|
|  Anil Chaudhary|     Engineering|      2.6|
|    Puja Koirala|        Business|      3.5|
|     Nabin Dahal|            Arts

In [10]:
# Q8. Show all students from the Computer Science department who have a GPA above 3.5.

df_filled.filter(
    (col("department") == "Computer Science") & (col("gpa") > 3.5)
).show()

+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|student_id|            name|gender|age|      department|year|gpa|scholarship|     city|               email|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|       101|    Aarav Sharma|  Male| 20|Computer Science|   2|3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       104|        Sita Rai|Female| 19|Computer Science|   1|3.9|        Yes|Kathmandu|    sita.rai@uni.edu|
|       108|     Kavya Joshi|Female| 20|Computer Science|   2|3.7|        Yes|  Pokhara| kavya.joshi@uni.edu|
|       112|     Manisha Oli|Female| 21|Computer Science|   3|3.6|        Yes|  Pokhara| manisha.oli@uni.edu|
|       120|Kabita Bhattarai|Female| 22|Computer Science|   4|3.9|        Yes|Bhaktapur|kabita.bhattarai@...|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+



In [11]:
# Q9. Display students who are in year 3 or year 4 AND have a scholarship. Use isin() for the year condition.

df_filled.filter(
    col("year").isin(3, 4) & (col("scholarship") == "Yes")
).show()

+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|student_id|            name|gender|age|      department|year|gpa|scholarship|     city|               email|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|       111|   Prakash Bista|  Male| 24|        Business|   4|3.4|        Yes|Kathmandu|prakash.bista@uni...|
|       112|     Manisha Oli|Female| 21|Computer Science|   3|3.6|        Yes|  Pokhara| manisha.oli@uni.edu|
|       116|     Rupa Basnet|Female| 23|      Undeclared|   4|3.7|        Yes|  Pokhara| rupa.basnet@uni.edu|
|       120|Kabita Bhattarai|Female| 22|Computer Science|   4|3.9|        Yes|Bhaktapur|kabita.bhattarai@...|
|       121|      Suman Giri|  Male| 21|     Engineering|   3|3.4|        Yes|Kathmandu|  suman.giri@uni.edu|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+



In [12]:
# Q10. Add a new column grade: Distinction if GPA >= 3.7, Merit if GPA >= 3.0, else Pass.

df_filled = df_filled.withColumn("grade",
    when(col("gpa") >= 3.7, "Distinction")
    .when(col("gpa") >= 3.0, "Merit")
    .otherwise("Pass")
)

df_filled.show()

+----------+----------------+------+---+----------------+----+---------+-----------+---------+--------------------+-----------+
|student_id|            name|gender|age|      department|year|      gpa|scholarship|     city|               email|      grade|
+----------+----------------+------+---+----------------+----+---------+-----------+---------+--------------------+-----------+
|       101|    Aarav Sharma|  Male| 20|Computer Science|   2|      3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|Distinction|
|       102|     Priya Thapa|Female| 22|        Business|   4|      3.2|         No|  Pokhara| priya.thapa@uni.edu|      Merit|
|       103|     Rohan Karki|  Male| 21|     Engineering|   3|3.3045454|         No| Lalitpur| rohan.karki@uni.edu|      Merit|
|       104|        Sita Rai|Female| 19|Computer Science|   1|      3.9|        Yes|Kathmandu|    sita.rai@uni.edu|Distinction|
|       105|   Bikash Gurung|  Male| 23|        Business|   4|      2.7|         No|   Butwal|bikash.gur

In [13]:
# Q11. Drop the email and student_id columns from the DataFrame. Confirm by printing the column names.

df_dropped = df_filled.drop("email", "student_id")
print(df_dropped.columns)

['name', 'gender', 'age', 'department', 'year', 'gpa', 'scholarship', 'city', 'grade']


In [14]:
# Q12. Remove all rows where department is null, then remove duplicate rows based on name and department.

df_dropped = df_dropped.filter(col("department").isNotNull())
df_dropped = df_dropped.dropDuplicates(["name", "department"])
df_dropped.show()

+----------------+------+---+----------------+----+---------+-----------+---------+-----------+
|            name|gender|age|      department|year|      gpa|scholarship|     city|      grade|
+----------------+------+---+----------------+----+---------+-----------+---------+-----------+
|    Aarav Sharma|  Male| 20|Computer Science|   2|      3.8|        Yes|Kathmandu|Distinction|
|  Anil Chaudhary|  Male| 21|     Engineering|   3|      2.6|         No|Nepalgunj|       Pass|
|  Anita Shrestha|Female| 20|     Engineering|   2|      3.5|        Yes|Bhaktapur|      Merit|
|   Bikash Gurung|  Male| 23|        Business|   4|      2.7|         No|   Butwal|       Pass|
|  Bishal Ghimire|  Male| 23|     Engineering|   4|      2.8|         No|Kathmandu|       Pass|
|   Deepak Pandey|  Male| 22|            Arts|   3|      3.1|         No|Kathmandu|      Merit|
|  Dipika Niroula|Female| 20|        Business|   2|3.3045454|         No|   Dharan|      Merit|
|Kabita Bhattarai|Female| 22|Computer Sc

In [15]:
# Q13. Rename gpa to grade_point_avg and year to study_year using withColumnRenamed.

df_renamed = df_dropped.withColumnRenamed("gpa", "grade_point_avg") \
                       .withColumnRenamed("year", "study_year")

df_renamed.show()

+----------------+------+---+----------------+----------+---------------+-----------+---------+-----------+
|            name|gender|age|      department|study_year|grade_point_avg|scholarship|     city|      grade|
+----------------+------+---+----------------+----------+---------------+-----------+---------+-----------+
|    Aarav Sharma|  Male| 20|Computer Science|         2|            3.8|        Yes|Kathmandu|Distinction|
|  Anil Chaudhary|  Male| 21|     Engineering|         3|            2.6|         No|Nepalgunj|       Pass|
|  Anita Shrestha|Female| 20|     Engineering|         2|            3.5|        Yes|Bhaktapur|      Merit|
|   Bikash Gurung|  Male| 23|        Business|         4|            2.7|         No|   Butwal|       Pass|
|  Bishal Ghimire|  Male| 23|     Engineering|         4|            2.8|         No|Kathmandu|       Pass|
|   Deepak Pandey|  Male| 22|            Arts|         3|            3.1|         No|Kathmandu|      Merit|
|  Dipika Niroula|Female| 20

In [16]:
# Q14. Rename all columns at once by replacing underscores with spaces using toDF(). Print the new column names.

new_names = [c.replace("_", " ") for c in df_renamed.columns]
df_renamed.toDF(*new_names).show()
print(new_names)

+----------------+------+---+----------------+----------+---------------+-----------+---------+-----------+
|            name|gender|age|      department|study year|grade point avg|scholarship|     city|      grade|
+----------------+------+---+----------------+----------+---------------+-----------+---------+-----------+
|    Aarav Sharma|  Male| 20|Computer Science|         2|            3.8|        Yes|Kathmandu|Distinction|
|  Anil Chaudhary|  Male| 21|     Engineering|         3|            2.6|         No|Nepalgunj|       Pass|
|  Anita Shrestha|Female| 20|     Engineering|         2|            3.5|        Yes|Bhaktapur|      Merit|
|   Bikash Gurung|  Male| 23|        Business|         4|            2.7|         No|   Butwal|       Pass|
|  Bishal Ghimire|  Male| 23|     Engineering|         4|            2.8|         No|Kathmandu|       Pass|
|   Deepak Pandey|  Male| 22|            Arts|         3|            3.1|         No|Kathmandu|      Merit|
|  Dipika Niroula|Female| 20

In [17]:
# Q15. Run describe() on gpa and age. What is the mean GPA? What is the minimum age?

df_filled.describe("gpa", "age").show()

+-------+-------------------+------------------+
|summary|                gpa|               age|
+-------+-------------------+------------------+
|  count|                 25|                25|
|   mean|  3.304545450210571|              21.0|
| stddev|0.36909493967779344|1.3844373104863457|
|    min|                2.6|                19|
|    max|                3.9|                24|
+-------+-------------------+------------------+



In [18]:
# Q16. Find the average GPA per department. Sort the result from highest to lowest average GPA.

df_filled.groupBy("department") \
         .agg(F.avg("gpa").alias("avg_gpa")) \
         .orderBy(col("avg_gpa").desc()) \
         .show()

+----------------+------------------+
|      department|           avg_gpa|
+----------------+------------------+
|      Undeclared| 3.700000047683716|
|Computer Science| 3.614285707473755|
|        Business|3.2209091186523438|
|            Arts|3.2009090423583983|
|     Engineering| 3.072077921458653|
+----------------+------------------+



In [19]:
# Q17. Count how many students are on scholarship vs not, broken down by gender.

df_filled.groupBy("gender", "scholarship") \
         .agg(F.count("*").alias("student_count")) \
         .orderBy("gender", "scholarship") \
         .show()

+------+-----------+-------------+
|gender|scholarship|student_count|
+------+-----------+-------------+
|Female|         No|            5|
|Female|        Yes|            7|
|  Male|         No|           10|
|  Male|        Yes|            3|
+------+-----------+-------------+



In [20]:
''' Q18. For each city, find the number of students, average GPA, and count of scholarship holders.
Display cities with more than 2 students only. '''

df_filled.groupBy("city").agg(
    F.count("*").alias("student_count"),
    F.avg("gpa").alias("avg_gpa"),
    F.sum(when(col("scholarship") == "Yes", 1).otherwise(0)).alias("scholarship_count")
).filter(col("student_count") > 2) \
 .orderBy(col("student_count").desc()) \
 .show()

+---------+-------------+------------------+-----------------+
|     city|student_count|           avg_gpa|scholarship_count|
+---------+-------------+------------------+-----------------+
|Kathmandu|            8|3.3375000059604645|                4|
|  Pokhara|            5|3.5599999904632567|                4|
| Lalitpur|            4| 3.227272689342499|                0|
+---------+-------------+------------------+-----------------+



Bonus Challenge — Full Cleaning Pipeline

In [21]:
# Step 1 — Drop rows where BOTH gpa and age are null
df_bonus = df.filter(~(col("gpa").isNull() & col("age").isNull()))

In [22]:
# Step 2 — Fill remaining null gpa values with the department average
dept_avg = df_bonus.groupBy("department").agg(mean("gpa").alias("dept_avg_gpa"))
df_bonus = df_bonus.join(dept_avg, on="department", how="left")
df_bonus = df_bonus.withColumn("gpa", when(col("gpa").isNull(), col("dept_avg_gpa")).otherwise(col("gpa")))
df_bonus = df_bonus.drop("dept_avg_gpa")

In [23]:
# Step 3 — Add grade column
df_bonus = df_bonus.withColumn("grade",
    when(col("gpa") >= 3.7, "Distinction")
    .when(col("gpa") >= 3.0, "Merit")
    .otherwise("Pass")
)

In [24]:
# Step 4 — Drop the email column
df_bonus = df_bonus.drop("email")

In [25]:
# Step 5 — Rename gpa to grade_point_avg
df_bonus = df_bonus.withColumnRenamed("gpa", "grade_point_avg")

In [26]:
# Step 6 — Sort by grade_point_avg descending
df_bonus = df_bonus.orderBy(col("grade_point_avg").desc())
df_bonus.show()

+----------------+----------+----------------+------+----+----+------------------+-----------+---------+-----------+
|      department|student_id|            name|gender| age|year|   grade_point_avg|scholarship|     city|      grade|
+----------------+----------+----------------+------+----+----+------------------+-----------+---------+-----------+
|Computer Science|       104|        Sita Rai|Female|  19|   1|3.9000000953674316|        Yes|Kathmandu|Distinction|
|Computer Science|       120|Kabita Bhattarai|Female|  22|   4|3.9000000953674316|        Yes|Bhaktapur|Distinction|
|Computer Science|       101|    Aarav Sharma|  Male|  20|   2| 3.799999952316284|        Yes|Kathmandu|Distinction|
|Computer Science|       108|     Kavya Joshi|Female|NULL|   2| 3.700000047683716|        Yes|  Pokhara|Distinction|
|            NULL|       116|     Rupa Basnet|Female|  23|   4| 3.700000047683716|        Yes|  Pokhara|Distinction|
|Computer Science|       112|     Manisha Oli|Female|  21|   3|3

In [27]:
# Step 7 — Save as Parquet file
df_bonus.write.parquet("output/students_clean", mode="overwrite")
print("Saved to output/students_clean as Parquet")

Saved to output/students_clean as Parquet



Output:

In [28]:
df_result = spark.read.parquet("output/students_clean")
df_result.show()

+----------------+----------+----------------+------+----+----+------------------+-----------+---------+-----------+
|      department|student_id|            name|gender| age|year|   grade_point_avg|scholarship|     city|      grade|
+----------------+----------+----------------+------+----+----+------------------+-----------+---------+-----------+
|Computer Science|       104|        Sita Rai|Female|  19|   1|3.9000000953674316|        Yes|Kathmandu|Distinction|
|Computer Science|       120|Kabita Bhattarai|Female|  22|   4|3.9000000953674316|        Yes|Bhaktapur|Distinction|
|Computer Science|       101|    Aarav Sharma|  Male|  20|   2| 3.799999952316284|        Yes|Kathmandu|Distinction|
|Computer Science|       108|     Kavya Joshi|Female|NULL|   2| 3.700000047683716|        Yes|  Pokhara|Distinction|
|            NULL|       116|     Rupa Basnet|Female|  23|   4| 3.700000047683716|        Yes|  Pokhara|Distinction|
|Computer Science|       112|     Manisha Oli|Female|  21|   3|3

In [29]:
spark.stop()